# Oral screen shift: the whole run in Colab

Downloads the four open datasets, trains every model and saves the predictions. Nothing is downloaded to your own computer except one small zip at the end.

Before running: **Runtime > Change runtime type > T4 GPU**.

If Colab disconnects, run the cells again from the top: finished work is skipped.

## 1. Code, Google Drive, GPU check

Drive stores the features and predictions (under 100 MB), so a disconnect does not lose them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
!rm -rf oral-screen-shift
!git clone -q https://github.com/sachchi-sapiens/oral-screen-shift.git
%cd /content/oral-screen-shift
!pip install -q timm

# keep everything that takes time to make on Drive
!mkdir -p /content/drive/MyDrive/oral-screen-shift/features /content/drive/MyDrive/oral-screen-shift/preds results
!ln -sfn /content/drive/MyDrive/oral-screen-shift/features features
!ln -sfn /content/drive/MyDrive/oral-screen-shift/preds results/preds

import torch
assert torch.cuda.is_available(), 'No GPU: set Runtime > Change runtime type > T4 GPU'
print(torch.cuda.get_device_name(0))

## 2. Check the code works here (about 2 minutes)

In [ ]:
!python -m pytest -q --deselect tests/test_experiments.py

## 3. Download the four open datasets (about 10 GB, onto Colab's own disk)

In [ ]:
for site in ['smartom', 'china', 'karnataka', 'code']:
    !python scripts/download.py --site {site} --out /content/data

## 4. Make small copies of the photos so training is fast

The train/validation/test splits are already in the repository (`splits/`), made by `scripts/prepare.py`.

In [ ]:
!python scripts/shrink.py --data /content/data --out /content/data_small

## 5. Features and linear probes

In [ ]:
!python -m experiments.run_gpu --data /content/data_small --stage features
!python -m experiments.run_gpu --data /content/data_small --stage probe

## 6. Baseline ResNet-50 (6 models)

In [ ]:
!python -m experiments.run_gpu --data /content/data_small --stage resnet

## 7. ResNet-50 with colour augmentation (9 models)

In [ ]:
!python -m experiments.run_gpu --data /content/data_small --stage resnet_color

## 8. Pack the results

This writes `oral-screen-shift-results.zip` to the `oral-screen-shift` folder in your Google Drive (a few MB). Download it and unzip it into the `results` folder of the project on your computer. The tables and figures are then made there with `python -m experiments.e1_matrix` and the other experiment scripts.

In [ ]:
!python -m experiments.e4_site
!cd results && rm -f /content/drive/MyDrive/oral-screen-shift/oral-screen-shift-results.zip && zip -q -r /content/drive/MyDrive/oral-screen-shift/oral-screen-shift-results.zip preds/*.npy e4_site.csv && ls -lh /content/drive/MyDrive/oral-screen-shift/oral-screen-shift-results.zip
!ls results/preds | wc -l